In [112]:
#%pip install numpy matplotlib pandas scipy

In [113]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import re

In [114]:
def read_data(file_path: str):
  data = pd.read_csv(file_path)
  return data

In [131]:
def process(file_path: str, powerjoular: bool):
  data = read_data(file_path)
  if powerjoular:
    data = data["CPU Power"]
  else:
    data = data["W"]
    
  q25, q75 = data.quantile([0.25, 0.75])
  iqr = q75 - q25
  data = data.between(q25 - 1.5*iqr, q75 + 1.5*iqr)
  filtered_data = data.sample(n=150)
  num_cpu = "".join(re.findall("\d", file_path))
  if num_cpu == '':
    num_cpu = 0
  return filtered_data, int(num_cpu)

In [132]:
import os

powerspy_files = os.listdir("./output")
powerjoular_files = os.listdir("./powerjoular_output")
powerjoular_files_k = os.listdir("./powerjoular_output_k")
powerjoular_glob_files = []
powerjoular_stress_ng_files = []
powerjoular_glob_files_k = []
powerjoular_stress_ng_files_k = []

for file in powerjoular_files:
  if "stress-ng" in file:
    powerjoular_stress_ng_files.append(file)
  else:
    powerjoular_glob_files.append(file)

for file in powerjoular_files_k:
  if "stress-ng" in file:
    powerjoular_stress_ng_files_k.append(file)
  else:
    powerjoular_glob_files_k.append(file)
    
print(powerjoular_glob_files)
print(powerjoular_stress_ng_files)
print(powerjoular_glob_files_k)
print(powerjoular_stress_ng_files_k)

['idle', 'powerjoular-cpu-2', 'powerjoular-cpu-3', 'powerjoular-cpu-4', 'powerjoular-cpu-1']
['powerjoular-cpu-4-stress-ng-cpu.csv', 'powerjoular-cpu-1-stress-ng-cpu.csv', 'powerjoular-cpu-3-stress-ng-cpu.csv', 'powerjoular-cpu-2-stress-ng-cpu.csv']
['powerjoular-cpu-1-k', 'powerjoular-cpu-2-k', 'powerjoular-cpu-3-k', 'powerjoular-cpu-4-k', 'idle_k']
['powerjoular-cpu-4-k-stress-ng-cpu.csv', 'powerjoular-cpu-2-k-stress-ng-cpu.csv', 'powerjoular-cpu-3-k-stress-ng-cpu.csv', 'powerjoular-cpu-1-k-stress-ng-cpu.csv']


In [133]:
def process_files(files: list[str], path: str, powerjoular: bool = True):
  data_dict = {}
  for f in files:
    df, num_cpu = process(f"{path}{f}", powerjoular)
    data_dict[num_cpu] = df
  return data_dict

In [134]:
powerspy_data = process_files(powerspy_files, "./output/", False)
powerjoular_glob_data = process_files(powerjoular_glob_files, "./powerjoular_output/")
powerjoular_stress_ng_data = process_files(powerjoular_stress_ng_files, "./powerjoular_output/")
powerjoular_glob_k_data = process_files(powerjoular_glob_files_k, "./powerjoular_output_k/")
powerjoular_stress_ng_k_data = process_files(powerjoular_stress_ng_files_k, "./powerjoular_output_k/")

In [125]:
def print_summary(title, data_dict):
  print(title)
  for k, v in data_dict.items():
    print(f"num cpu {k}, mean {v.mean()}")

In [126]:
print_summary("Powerspy summary", powerspy_data)
print_summary("Power joular global scope summary", powerjoular_glob_data)
print_summary("Power joular stress ng scope summary", powerjoular_stress_ng_data)
print_summary("Power joular using thread level stats global scope summary", powerjoular_glob_k_data)
print_summary("Power joular using thread level stats stress ng scope summary", powerjoular_stress_ng_k_data)

Powerspy summary
num cpu 0, mean 5.5409062005471
num cpu 1, mean 6.652643498241321
num cpu 2, mean 7.631913168962204
num cpu 3, mean 8.654758858387698
num cpu 4, mean 9.72339666875637
Power joular global scope summary
num cpu 0, mean 3.4009518752678347
num cpu 2, mean 4.6097507290267625
num cpu 3, mean 5.709952395047669
num cpu 4, mean 6.760827008778049
num cpu 1, mean 3.3208506513650295
Power joular stress ng scope summary
num cpu 4, mean 6.53883986524974
num cpu 1, mean 3.207158146895456
num cpu 3, mean 5.580907763455363
num cpu 2, mean 4.495949742930494
Power joular using thread level stats global scope summary
num cpu 1, mean 3.3210757550837338
num cpu 2, mean 4.610784807375415
num cpu 3, mean 5.706418404022642
num cpu 4, mean 6.760550994552635
num cpu 0, mean 3.400006600853807
Power joular using thread level stats stress ng scope summary
num cpu 4, mean 6.538419832858246
num cpu 2, mean 4.493727807345648
num cpu 3, mean 5.58361697733996
num cpu 1, mean 3.2054162408799294
